# Chapter 3 Practical 04: Item-Item Collaborative Filtering

Learning objectives:
- Compute item-item similarities from user ratings.
- Predict a missing rating from similar items the user already rated.
- Compare item-item CF with user-user CF.
- Discuss why item-item CF is often easier to cache and serve.

Slide connection: item-item CF concept, item-item prediction example, and scalability.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Item-item CF compares columns of the user-item matrix. Two movies are compared only using users who rated both movies.


In [ ]:
def item_pearson(matrix, item_a, item_b):
    pair = matrix[[item_a, item_b]].dropna()
    if len(pair) < 2:
        return np.nan
    if pair[item_a].std() == 0 or pair[item_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair[item_a], pair[item_b])[0, 1])

items = rating_matrix.columns
item_sim = pd.DataFrame(index=items, columns=items, dtype=float)
for a in items:
    for b in items:
        item_sim.loc[a, b] = 1.0 if a == b else item_pearson(rating_matrix, a, b)

item_sim.round(2)


For one target item, inspect which other movies are most similar based on shared user ratings.


In [ ]:
target_item = "Independence Day"
item_sim[target_item].drop(target_item).sort_values(ascending=False).round(3)


To predict a user's missing rating, use similar items the same user has already rated. Negative item similarities are skipped in the beginner version because they indicate opposite rating patterns.


In [ ]:
def predict_item_item(matrix, target_user, target_item, k_neighbors=3, positive_only=True):
    rated = matrix.loc[target_user].dropna()
    candidates = []
    for item, rating in rated.items():
        sim = item_sim.loc[target_item, item]
        if pd.isna(sim):
            continue
        if positive_only and sim <= 0:
            continue
        candidates.append({"rated_item": item, "rating": rating, "similarity": sim})
    evidence = pd.DataFrame(candidates, columns=["rated_item", "rating", "similarity"])
    evidence = evidence.sort_values("similarity", ascending=False).head(k_neighbors)
    if evidence.empty:
        return np.nan, evidence
    numerator = (evidence["rating"] * evidence["similarity"]).sum()
    denominator = evidence["similarity"].abs().sum()
    return numerator / denominator if denominator else np.nan, evidence

target_user = "Karen"
k_neighbors = 3

pred, evidence = predict_item_item(rating_matrix, target_user, target_item, k_neighbors=k_neighbors)
print(f"Predicted Karen rating for Independence Day: {pred:.2f}")
evidence.round(3)


The recommender applies the same item-item prediction to every unseen movie and returns the highest predicted ratings.


In [ ]:
def recommend_item_item(matrix, target_user, n=5, k_neighbors=3):
    unseen_items = matrix.columns[matrix.loc[target_user].isna()]
    rows = []
    for item in unseen_items:
        pred, evidence = predict_item_item(matrix, target_user, item, k_neighbors=k_neighbors, positive_only=True)
        if not pd.isna(pred):
            rows.append({
                "user": target_user,
                "recommended_movie": item,
                "predicted_rating": pred,
                "similar_rated_items": ", ".join(evidence["rated_item"].tolist()),
            })
    return pd.DataFrame(rows).sort_values("predicted_rating", ascending=False).head(n)

recommend_item_item(rating_matrix, "Karen").round(2)


# Challenges

### Challenge 1 — Change the Target Item

**Goal:**
Investigate how item-item evidence changes when the target item changes.

**What to do:**

1. Change `target_item` from `"Independence Day"` to another unseen item for Karen.
2. Rerun the item similarity and prediction cells.
3. Inspect the `evidence` table.
4. Compare the similar rated items with the original result.


In [ ]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which rated items supported the new prediction? Were the similarities based on enough shared users? Did the predicted rating increase or decrease?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Include Negative Item Similarities

**Goal:**
Investigate how negative item similarity can affect item-item predictions.

**What to do:**

1. In `predict_item_item`, call the function with `positive_only=False`.
2. Rerun the prediction for the same `target_user` and `target_item`.
3. Compare the evidence table with the positive-only version.
4. Explain whether the negative similarities made the result easier or harder to interpret.


In [ ]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Did any negative similarities appear? How did the predicted rating change? Would you keep or exclude negative similarities for a beginner recommender?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Item-Item Scalability

This challenge requires **no programming**.

Many platforms have millions of users but a smaller and more stable catalog of items.

Explain why item-item collaborative filtering can be easier to cache and serve than comparing a target user with every other user.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
